# Lab 14: Claims and reconciliation

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /private/var/folders/19/xhrx71p90m96kh792dr3hmlr0000gn/T/lab-ci-9jp6r49f/repo | LLM mode: replay


In [2]:
import inspect
print(inspect.getsource(schemas.MgmtClaim))
blocks = {r["block_id"]: r for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # block ids are unique only within a document
LARGE = llm.P["llm"]["models"]["large"]
extracted = {b: llm.structured(schemas.MgmtClaims, prompts.claims_messages(blocks[b]["text"]), LARGE) for b in sd.CLAIM_BLOCKS}
claims = sd.claim_rows(extracted)          # claim ids; CY2025 aligned to FY2025 with basis "calendar"
Path("outputs/work").mkdir(parents=True, exist_ok=True)
with open("outputs/work/claims.jsonl", "w") as fh:
    for c in claims: fh.write(json.dumps(c) + "\n")
import pandas as pd
pd.DataFrame(claims)[["claim_id", "metric", "value_abs", "period_label", "period_basis", "qualifier"]]

class MgmtClaim(BaseModel):                       # Lab 14
    metric: str = Field(description="vocab metric name")
    value: float
    unit: Literal["USD", "pct", "count", "ratio"]
    scale: Literal[1, 1_000, 1_000_000, 1_000_000_000]
    period: str = Field(description="e.g. FY2025, LTM")
    qualifier: Literal["reported", "adjusted", "run-rate", "pro forma", "LTM", "FY"]
    quote: str = Field(description="exact source span")



,claim_id,metric,value_abs,period_label,period_basis,qualifier
0,CL-DR-deal_a-001-p0002_b001-0,revenue_growth,0.32,FY2025,fiscal,reported
1,CL-DR-deal_a-001-p0002_b001-1,revenue,41200000.00,FY2025,fiscal,reported
2,CL-DR-deal_a-001-p0002_b002-0,revenue,48000000.00,FY2025,fiscal,run-rate
3,CL-DR-deal_a-001-p0002_b003-0,revenue,43500000.00,CY2025,calendar,reported
4,CL-DR-deal_a-001-p0003_b001-0,top1_customer_share,0.08,FY2025,fiscal,reported
5,CL-DR-deal_a-001-p0003_b002-0,nrr,1.18,FY2025,fiscal,reported


In [3]:
# From the slide: Reconciliation: claims against seller financials
def reconcile(claim, facts, tol=0.01, high=0.05):
    """Tier-4 claim vs tier-3 facts -> output record."""
    rec = {"metric": claim.metric, "period": claim.period,
           "claim_id": claim.claim_id, "fact_id": None,
           "status": None, "gap": None, "severity": None}
    f = facts.get((claim.metric, claim.period))
    if f is None:
        rec["status"] = "no_fact"; return rec
    rec["fact_id"] = f["fact_id"]
    if claim.qualifier in ("run-rate", "pro forma"):
        rec["status"] = "definition_mismatch"; return rec
    if claim.period_basis != f["period_basis"]:
        rec["status"] = "period_mismatch"; return rec
    v = claim.value * claim.scale
    gap = (v - f["value"]) / abs(f["value"])
    rec["gap"] = round(gap, 6)
    if abs(gap) <= tol:
        rec["status"] = "match"; return rec
    rec["status"] = "discrepancy"
    rec["severity"] = "high" if abs(gap) > high else "medium"
    return rec

In [4]:
from types import SimpleNamespace as NS
facts = {(f["metric"], f["period"]): f for f in sd.seller_facts()}
R = llm.P["reconcile"]
recs = [reconcile(NS(**c), facts, tol=R["tolerance_rel"], high=R["high_gap_rel"]) for c in claims]
Path("outputs/reconcile").mkdir(parents=True, exist_ok=True)
with open("outputs/reconcile/deal_a.jsonl", "w") as fh:     # the CS2 section-8 format
    for r in recs: fh.write(json.dumps(r) + "\n")
pd.DataFrame(recs)

,metric,period,claim_id,fact_id,status,gap,severity
0,revenue_growth,FY2025,CL-DR-deal_a-001-p0002_b001-0,F-3e153fb7cf,discrepancy,0.301649,high
1,revenue,FY2025,CL-DR-deal_a-001-p0002_b001-1,F-b9733fc812,match,0.000000,NaN
2,revenue,FY2025,CL-DR-deal_a-001-p0002_b002-0,F-b9733fc812,definition_mismatch,NaN,NaN
3,revenue,FY2025,CL-DR-deal_a-001-p0002_b003-0,F-b9733fc812,period_mismatch,NaN,NaN
4,top1_customer_share,FY2025,CL-DR-deal_a-001-p0003_b001-0,F-6055a5ac1f,discrepancy,-0.448276,high
5,nrr,FY2025,CL-DR-deal_a-001-p0003_b002-0,F-69cbc9e5f9,discrepancy,0.082569,high


In [5]:
import yaml
key = yaml.safe_load(open("data/dataroom/deal_a/answer_key.yaml"))
expected = {t["claim_block"]: t["expected_status"] for t in key["traps"]}
by_block = {c["block_id"]: r for c, r in zip(claims, recs) if c["block_id"] in expected and r["status"] != "match"}
hits = sum(by_block.get(b, {}).get("status") == s for b, s in expected.items())
flagged = sum(r["status"] != "match" for r in recs)
print(f"trap recall {hits}/{len(expected)} | precision {hits}/{flagged}")

trap recall 5/5 | precision 5/5


**Exercise 14.** Why does the run-rate claim come back `definition_mismatch` rather than `discrepancy`? Change the reconcile order so the gap is computed first: what goes wrong?

In [6]:
def reconcile_gap_first(claim, facts, tol=0.01, high=0.05):
    rec = {
        "metric": claim.metric,
        "period": claim.period,
        "claim_id": claim.claim_id,
        "fact_id": None,
        "status": None,
        "gap": None,
        "severity": None,
    }

    f = facts.get((claim.metric, claim.period))
    if f is None:
        rec["status"] = "no_fact"
        return rec

    rec["fact_id"] = f["fact_id"]

    # MOVED UP: classify the numerical difference first.
    v = claim.value * claim.scale
    gap = (v - f["value"]) / abs(f["value"])
    rec["gap"] = round(gap, 6)

    if abs(gap) > tol:
        rec["status"] = "discrepancy"
        rec["severity"] = "high" if abs(gap) > high else "medium"
        return rec  # BUG: skips the checks below!

    # These checks now happen too late.
    if claim.qualifier in ("run-rate", "pro forma"):
        rec["status"] = "definition_mismatch"
        return rec

    if claim.period_basis != f["period_basis"]:
        rec["status"] = "period_mismatch"
        return rec

    rec["status"] = "match"
    return rec

In [7]:
recs_gap_first = [
    reconcile_gap_first(
        NS(**c),
        facts,
        tol=R["tolerance_rel"],
        high=R["high_gap_rel"],
    )
    for c in claims
]

display(pd.DataFrame(recs_gap_first))

,metric,period,claim_id,fact_id,status,gap,severity
0,revenue_growth,FY2025,CL-DR-deal_a-001-p0002_b001-0,F-3e153fb7cf,discrepancy,0.301649,high
1,revenue,FY2025,CL-DR-deal_a-001-p0002_b001-1,F-b9733fc812,match,0.000000,NaN
2,revenue,FY2025,CL-DR-deal_a-001-p0002_b002-0,F-b9733fc812,discrepancy,0.165049,high
3,revenue,FY2025,CL-DR-deal_a-001-p0002_b003-0,F-b9733fc812,discrepancy,0.055825,high
4,top1_customer_share,FY2025,CL-DR-deal_a-001-p0003_b001-0,F-6055a5ac1f,discrepancy,-0.448276,high
5,nrr,FY2025,CL-DR-deal_a-001-p0003_b002-0,F-69cbc9e5f9,discrepancy,0.082569,high


In [8]:
# Score the intentionally incorrect order against the same answer key.
by_block_gap = {c["block_id"]: r for c, r in zip(claims, recs_gap_first)
                if c["block_id"] in expected and r["status"] != "match"}
hits_gap = sum(by_block_gap.get(b, {}).get("status") == status
               for b, status in expected.items())
flagged_gap = sum(r["status"] != "match" for r in recs_gap_first)
print(f"gap-first trap recall {hits_gap}/{len(expected)} | precision {hits_gap}/{flagged_gap}")
assert hits == flagged == len(expected) == 5
assert hits_gap == 3 and flagged_gap == 5
assert by_block["p0002_b002"]["status"] == "definition_mismatch"
assert by_block["p0002_b003"]["status"] == "period_mismatch"
assert by_block_gap["p0002_b002"]["status"] == "discrepancy"
assert by_block_gap["p0002_b003"]["status"] == "discrepancy"


gap-first trap recall 3/5 | precision 3/5


The run-rate figure uses a different definition from reported revenue, while CY2025 covers a different period from FY2025. Check these before computing a gap. The correct order identifies all five traps (5/5 recall and precision). Computing the gap first incorrectly labels W2 and W3 as discrepancies, reducing both scores to 3/5. The intentionally incorrect function is used only for this exercise; saved reconciliation outputs use `reconcile`.